In [4]:
%pip install beautifulsoup4
%pip install contractions
%pip install joblib
%pip install kaleido
%pip install langdetect
%pip install matplotlib
%pip install nltk
%pip install numpy
%pip install optuna
%pip install pandas
%pip install plotly
%pip install scikit-learn
%pip install seaborn
%pip install spacy
%pip install spacymoji
%pip install wordcloud

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated pa

In [5]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import pickle
import re
import string
import sys
import time
import unicodedata
from collections import Counter

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import spacymoji
from mpl_toolkits.mplot3d import Axes3D
from spacy import displacy
from spacy.lang.fr import French
from spacy.tokenizer import _get_regex_pattern
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objs as go
import plotly.graph_objects as go
import plotly.io as pio
import plotly.offline as py
from plotly.subplots import make_subplots

import nltk
from nltk import pos_tag, sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.stem.snowball import SnowballStemmer
nltk.download("averaged_perceptron_tagger")
nltk.download("averaged_perceptron_tagger_eng")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("tagsets")
nltk.download("tagsets_json")
nltk.download("wordnet")

import sklearn
from sklearn import metrics
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

import optuna
import contractions
from bs4 import BeautifulSoup
from langdetect import detect

!python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm")

stop_words = set(stopwords.words("english"))

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/e20210005280/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/e20210005280/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/e20210005280/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/e20210005280/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/e20210005280/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package tagsets to
[nltk_data]     /home/e20210005280/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 5.4 MB/s  0:00:02m0:00:0100:01m
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 🚀 Run from here

In [6]:
df=pd.read_csv('scitweets_export.tsv',sep='\t')
print (df.head())

   Unnamed: 0            tweet_id  \
0           0  316669998137483264   
1           1  319090866545385472   
2           2  322030931022065664   
3           3  322694830620807168   
4           4  328524426658328576   

                                                text  science_related  \
0  Knees are a bit sore. i guess that's a sign th...                0   
1          McDonald's breakfast stop then the gym 🏀💪                0   
2  Can any Gynecologist with Cancer Experience ex...                1   
3  Couch-lock highs lead to sleeping in the couch...                1   
4  Does daily routine help prevent problems with ...                1   

   scientific_claim  scientific_reference  scientific_context  
0               0.0                   0.0                 0.0  
1               0.0                   0.0                 0.0  
2               1.0                   0.0                 0.0  
3               1.0                   0.0                 0.0  
4               1.

In [7]:
def normalize(X,
              removelowercase=False,
              removestopwords=False,
              removedigit=False,
              removeemoji=False,
              getstemmer=False,
              getlemmatisation=False):
    
    nlp = spacy.load("en_core_web_sm")
    nlp.add_pipe("emoji", first=True)
    re_token_match = spacy.tokenizer._get_regex_pattern(nlp.Defaults.token_match)
    re_token_match = f"({re_token_match}|#\w+|\w+-\w+)"
    X = contractions.fix(X)
    
    if removelowercase:
        X = X.lower()
    
    doc = nlp(X)
    
    lem = WordNetLemmatizer()
    stemmer = PorterStemmer()
    
    tokens = []
    
    for t in doc:
        if t.is_punct or t.is_space:
            continue
        if removestopwords and t.is_stop:
            continue
        if removedigit and t.like_num:
            continue
        if removeemoji and t._.is_emoji:
            continue
        
        word = t.text
        if getlemmatisation:
            word = lem.lemmatize(word)
        if getstemmer:
            word = stemmer.stem(word)
            
        tokens.append(word)
    
    return " ".join(tokens)   

In [8]:
class TextNormalizer(BaseEstimator, TransformerMixin):
    def __init__(self,
              removelowercase=False,
              removestopwords=False,
              removedigit=False,
              removeemoji=False,
              getstemmer=False,
              getlemmatisation=False):
        
        self.removelowercase = removelowercase
        self.removestopwords = removestopwords
        self.removedigit = removedigit
        self.removeemoji = removeemoji
        self.getstemmer = getstemmer
        self.getlemmatisation = getlemmatisation
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        return [
            normalize(text,
            removelowercase=self.removelowercase,
            removestopwords=self.removestopwords,
            removedigit=self.removedigit,
            removeemoji=self.removeemoji,
            getstemmer=self.getstemmer,
            getlemmatisation=self.getlemmatisation)
            for text in X
        ]

In [16]:
models = [
    ("KNeighbors", KNeighborsClassifier()),
    ("DecisionTree", DecisionTreeClassifier()),
    ("SVC", SVC()),
    ("RandomForest", RandomForestClassifier())
]

In [28]:
X = df.text
y = df.science_related

for name,model in models : 
    pipe = Pipeline([
        ("cleaner", TextNormalizer()),
        ("vectorizer", TfidfVectorizer()),
        (name,model)
    ])

    pipe.fit(X,y)

    print("Apprentissage terminé sur l'ensemble du corpus.")

    model_path = "SentimentModel_"+name+".pkl"
    with open(model_path, "wb") as f:
        pickle.dump(pipe, f)


Apprentissage terminé sur l'ensemble du corpus.
Apprentissage terminé sur l'ensemble du corpus.
Apprentissage terminé sur l'ensemble du corpus.
Apprentissage terminé sur l'ensemble du corpus.


In [30]:
for name,model in models : 
    
    model_path = "SentimentModel_"+name+".pkl"
    with open(model_path, "rb") as f:
        sentiment_model = pickle.load(f)

    print("Modèle chargé :", model_path, "\n")

    new_review = "Covid is chinese made"
    # Prédiction
    prediction = sentiment_model.predict([new_review])[0]

    print("Avis :", new_review)
    print("Prédiction :", prediction)

Modèle chargé : SentimentModel_KNeighbors.pkl 

Avis : Covid is chinese made
Prédiction : 0
Modèle chargé : SentimentModel_DecisionTree.pkl 

Avis : Covid is chinese made
Prédiction : 0
Modèle chargé : SentimentModel_SVC.pkl 

Avis : Covid is chinese made
Prédiction : 0
Modèle chargé : SentimentModel_RandomForest.pkl 

Avis : Covid is chinese made
Prédiction : 0
